# Observability & Debugging — Part 4: Backend Export

This notebook teaches you how to export traces to production observability backends
using the OTLP (OpenTelemetry Protocol) exporter.

**What you'll learn:**
- Export traces to CloudWatch via AWS Distro for OpenTelemetry (ADOT)
- Export traces to Langfuse for LLM-specific observability
- Export traces to Jaeger for local development
- Switch backends with a single environment variable change

**Prerequisites:**
- Complete notebooks 01–03 first
- *(Optional)* Docker for running Jaeger locally
- *(Optional)* Langfuse account for cloud export
- *(Optional)* AWS credentials with CloudWatch permissions

In [ ]:
import os

from strands import Agent, tool
from strands.models.bedrock import BedrockModel
from strands.telemetry.config import StrandsTelemetry

print("✓ Imports ready")

## How OTLP Export Works

The OpenTelemetry Protocol (OTLP) is the standard wire format for sending telemetry
data. Strands provides `setup_otlp_exporter()` which reads the endpoint from the
`OTEL_EXPORTER_OTLP_ENDPOINT` environment variable.

The same exporter works with **any** OTLP-compatible backend:
- CloudWatch (via ADOT collector)
- Langfuse (direct OTLP ingestion)
- Jaeger (built-in OTLP receiver)
- Grafana Tempo, Honeycomb, Datadog, etc.

## Option 1: CloudWatch via ADOT

AWS Distro for OpenTelemetry (ADOT) is a collector that receives OTLP traces and
forwards them to CloudWatch X-Ray.

**Start the ADOT collector:**
```bash
docker run --rm -p 4318:4318 \
  -e AWS_REGION=us-east-1 \
  public.ecr.aws/aws-observability/aws-otel-collector:latest
```

In [ ]:
def setup_cloudwatch_telemetry():
    """Configure telemetry to export to CloudWatch via ADOT.

    Requires ADOT collector running at localhost:4318.
    Start with: docker run --rm -p 4318:4318 public.ecr.aws/aws-observability/aws-otel-collector:latest
    """
    os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4318")

    telemetry = StrandsTelemetry()
    telemetry.setup_console_exporter()  # Keep console for local visibility

    try:
        telemetry.setup_otlp_exporter()
        print("✓ CloudWatch telemetry configured (via ADOT at localhost:4318)")
        print("  Traces will appear in CloudWatch X-Ray console.")
    except Exception as e:
        print(f"⚠️  ADOT collector not available: {e}")
        print("   Traces will only appear in console output.")
        print("   Start ADOT: docker run --rm -p 4318:4318 public.ecr.aws/aws-observability/aws-otel-collector:latest")

    return telemetry


# Uncomment to activate:
# telemetry = setup_cloudwatch_telemetry()
print("CloudWatch setup defined. Uncomment the line above to activate.")

## Option 2: Langfuse

Langfuse provides a purpose-built UI for LLM observability with native OTLP ingestion.

**Setup:**
1. Create account at [langfuse.com](https://langfuse.com)
2. Get public/secret keys from Project Settings
3. Set environment variables below

In [ ]:
def setup_langfuse_telemetry():
    """Configure telemetry to export to Langfuse.

    Requires LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY environment variables.
    Get keys from: https://cloud.langfuse.com → Project Settings
    """
    host = os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com")
    public_key = os.environ.get("LANGFUSE_PUBLIC_KEY", "")
    secret_key = os.environ.get("LANGFUSE_SECRET_KEY", "")

    if not public_key or not secret_key:
        print("⚠️  LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY not set.")
        print("   Set these environment variables to enable Langfuse export:")
        print('   export LANGFUSE_PUBLIC_KEY="pk-lf-..."')
        print('   export LANGFUSE_SECRET_KEY="sk-lf-..."')
        return None

    os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = f"{host}/api/public/otel"
    os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = (
        f"Authorization=Basic {public_key}:{secret_key}"
    )

    telemetry = StrandsTelemetry()
    telemetry.setup_console_exporter()
    telemetry.setup_otlp_exporter()
    print(f"✓ Langfuse telemetry configured (host: {host})")
    print("  Traces will appear in your Langfuse project dashboard.")
    return telemetry


# Uncomment to activate:
# telemetry = setup_langfuse_telemetry()
print("Langfuse setup defined. Uncomment the line above to activate.")

## Option 3: Jaeger (Local Development)

Jaeger is ideal for local development — single Docker container with a built-in UI.

**Start Jaeger:**
```bash
docker run --rm -d --name jaeger \
  -p 4318:4318 \
  -p 16686:16686 \
  jaegertracing/all-in-one:latest
```

Then open http://localhost:16686 to view traces.

In [ ]:
def setup_jaeger_telemetry():
    """Configure telemetry to export to Jaeger.

    Requires Jaeger running at localhost:4318.
    Start with: docker run --rm -d -p 4318:4318 -p 16686:16686 jaegertracing/all-in-one:latest
    View traces at: http://localhost:16686
    """
    os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4318")

    telemetry = StrandsTelemetry()
    telemetry.setup_console_exporter()

    try:
        telemetry.setup_otlp_exporter()
        print("✓ Jaeger telemetry configured (localhost:4318)")
        print("  View traces at: http://localhost:16686")
    except Exception as e:
        print(f"⚠️  Jaeger not available: {e}")
        print("   Start: docker run --rm -d -p 4318:4318 -p 16686:16686 jaegertracing/all-in-one:latest")

    return telemetry


# Uncomment to activate:
# telemetry = setup_jaeger_telemetry()
print("Jaeger setup defined. Uncomment the line above to activate.")

## Testing Your Backend

Once you've uncommented one of the setup functions above, run the cell below
to send a test trace to your backend.

In [ ]:
# Use console-only telemetry for this demo (replace with your backend above)
telemetry = StrandsTelemetry()
telemetry.setup_console_exporter()


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


agent = Agent(
    model=BedrockModel(model_id="us.amazon.nova-lite-v1:0"),
    tools=[calculator],
)

print("Sending test trace...\n")
result = agent("What is 99 * 77?")
print(f"\nResult: {result}")
print("\n✓ Check your backend for the trace!")

## Summary

| Backend | Endpoint | Use Case |
|---------|----------|----------|
| Console | stdout | Development, quick debugging |
| CloudWatch (ADOT) | `localhost:4318` | AWS production monitoring |
| Langfuse | `cloud.langfuse.com/api/public/otel` | LLM-specific observability |
| Jaeger | `localhost:4318` | Local development with UI |

All backends use the same `setup_otlp_exporter()` call — only the endpoint changes.

## Next

Continue to [05_custom_metrics.ipynb](05_custom_metrics.ipynb) to learn how to add
custom span attributes, metrics, and production best practices.